# 4d. Net and gross revenue retention

**What an investor reads into this.** NRR answers "if Northbridge stopped selling to new customers today, how much would last year's customer base be worth in a year?" Above 100% the existing base grows on its own; below 100% the business has to win new customers just to stand still. GRR removes expansion and shows the pure loss rate (churn and downgrades), which is the floor under the revenue a buyer is paying for. Both are read alongside price increases: a price rise lifts NRR without any change in customer behaviour.

**Limit:** NRR and GRR can only be computed for 2023, 2024 and 2025. The data starts in January 2022, so there is no December 2021 opening point.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nb_utils import connect, q, sql_file, save_table, pct, ROOT, DOCS
import charts as ch

pd.set_option("display.width", 220)
con = connect()
dec = sql_file(con, "dec_mrr_customer_line").groupby(["customer_id", "year"], as_index=False).dec_mrr.sum()
k = sql_file(con, "uplift_factors").set_index("year").k
years = [y for y in sorted(dec.year.unique()) if y - 1 in dec.year.values]

def nrr_grr(y, exclude=(), deflate=False):
    o = dec[(dec.year == y - 1) & ~dec.customer_id.isin(exclude)].set_index("customer_id").dec_mrr
    c = dec[dec.year == y].set_index("customer_id").dec_mrr.reindex(o.index, fill_value=0.0)
    if deflate:
        c = c / float(k[y])
    return {"opening_mrr": o.sum(), "closing_mrr_same_customers": c.sum(), "closing_mrr_capped": np.minimum(c, o).sum(),
            "nrr_pct": 100 * c.sum() / o.sum(), "grr_pct": 100 * np.minimum(c, o).sum() / o.sum(),
            "customers_opening": len(o), "customers_retained": int((c > 0).sum())}

## 1. NRR and GRR, with sensitivities

- **Excluding the top 5:** the five customers with the largest opening-December MRR are removed from both numerator and denominator.
- **Excluding the price increase:** closing MRR is divided by the year's uniform uplift factor (1.00, 1.05, 1.07; see `docs/metric_definitions.md`).

In [2]:
rows = []
for y in years:
    top5 = dec[dec.year == y - 1].sort_values(["dec_mrr", "customer_id"], ascending=[False, True]).head(5).customer_id.tolist()
    for name, kw in [("reported", {}), ("excluding top 5 customers", {"exclude": top5}),
                     ("excluding price increase", {"deflate": True}),
                     ("excluding top 5 and price increase", {"exclude": top5, "deflate": True})]:
        rows.append({"year": y, "basis": name, "uplift_factor": float(k[y]), **nrr_grr(y, **kw)})
nrr = save_table(pd.DataFrame(rows), "4d_nrr_grr")
nrr.pivot(index="basis", columns="year", values=["nrr_pct", "grr_pct"]).round(1)

nrr_pct             grr_pct            
year                                  2023  2024  2025    2023  2024  2025
basis                                                                     
excluding price increase              95.8  94.0  89.2    94.5  92.7  88.2
excluding top 5 and price increase    94.0  92.8  90.9    92.3  91.0  89.6
excluding top 5 customers             94.0  97.5  97.3    92.3  91.1  89.7
reported                              95.8  98.7  95.4    94.5  92.9  88.8

## 2. What good looks like: sourced benchmarks

The benchmark figures are transcribed from four published surveys into `data/reference/benchmarks_source.csv`, one row per source, segment, metric and statistic, with the URL and the date each report was opened (decision D12, resolved by D35 to D39). This notebook checks the file and copies it to `outputs/tables/benchmarks.csv`; nothing is fetched from the web during a run. Sources, definitions and caveats are in `docs/benchmarks.md`, which the last cell writes.

**Segments.** The surveys band companies by annual contract value (ACV, in US dollars). Bands under $50k are treated as SMB, $50k to $250k as mid-market and above $250k as enterprise. ChartMogul bands by monthly revenue per account (ARPA) instead.

**The SMB comparison.** For each measure, the range runs from the lowest to the highest SMB median in the latest edition of each source rated high or medium for comparability (SaaS Capital, High Alpha and Benchmarkit, all 2025). SaaS Capital is shown on its own as well, because its definitions match ours.


In [3]:
COLS = ["source_id", "publisher", "title", "year", "url", "accessed_date", "segment", "metric", "statistic", "value",
        "definition_notes", "comparability"]
src = pd.read_csv(ROOT / "data" / "reference" / "benchmarks_source.csv", dtype={"year": int, "value": float})
assert list(src.columns) == COLS
assert src.url.str.startswith("https://").all() and src.accessed_date.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()
assert src.statistic.isin(["median", "lower_quartile", "upper_quartile", "range_low", "range_high"]).all()
assert src.comparability.isin(["high", "medium", "low"]).all() and src.metric.isin(["NRR", "GRR", "Logo retention"]).all()
assert not src.duplicated(["source_id", "segment", "metric", "statistic"]).any()
bench = save_table(src, "benchmarks")

# Northbridge, 2025 (December 2024 to December 2025)
r25 = nrr[(nrr.year == 2025) & (nrr.basis == "reported")].iloc[0]
x25 = nrr[(nrr.year == 2025) & (nrr.basis == "excluding price increase")].iloc[0]
nb_vals = {"NRR": r25.nrr_pct, "NRR before price increase": x25.nrr_pct, "GRR": r25.grr_pct,
           "Logo retention": 100 * r25.customers_retained / r25.customers_opening}
acv = 12 * r25.opening_mrr / r25.customers_opening
acv_tbl = save_table(pd.DataFrame([{"year": 2025, "opening_mrr": r25.opening_mrr, "customers_opening": int(r25.customers_opening),
                                    "average_acv_gbp": acv, "average_arpa_gbp_month": acv / 12,
                                    "gbp_usd_rate_for_25k_usd": 25000 / acv, "gbp_usd_rate_for_50k_usd": 50000 / acv}]), "4d_northbridge_acv")

latest = bench.year == bench.groupby("publisher").year.transform("max")
smb = bench.segment.str.startswith("SMB:")
med = bench.statistic == "median"
comparable = bench.comparability.isin(["high", "medium"])
band = bench.segment == "SMB: ACV $25k to $50k"

def pick(mask):
    d = bench[mask]
    assert len(d), "no benchmark rows selected"
    return d

specs = [  # key, measure, benchmark basis, benchmark metric, rows
    ("smb", "NRR", "SMB medians, ACV under $50k (SaaS Capital, High Alpha, Benchmarkit 2025)", "NRR", latest & comparable & smb & med),
    ("smb", "NRR before price increase", "SMB medians, ACV under $50k (SaaS Capital, High Alpha, Benchmarkit 2025)", "NRR", latest & comparable & smb & med),
    ("smb", "GRR", "SMB medians, ACV under $50k (SaaS Capital, High Alpha, Benchmarkit 2025)", "GRR", latest & comparable & smb & med),
    ("sc_smb", "NRR", "SaaS Capital 2025 SMB medians (same definition as ours)", "NRR", (bench.source_id == "SC25") & smb & med),
    ("sc_smb", "NRR before price increase", "SaaS Capital 2025 SMB medians (same definition as ours)", "NRR", (bench.source_id == "SC25") & smb & med),
    ("sc_smb", "GRR", "SaaS Capital 2025 SMB medians (same definition as ours)", "GRR", (bench.source_id == "SC25") & smb & med),
    ("band", "NRR", "Medians for ACV $25k to $50k (SaaS Capital, High Alpha, Benchmarkit 2025)", "NRR", latest & comparable & band & med),
    ("band", "GRR", "Medians for ACV $25k to $50k (SaaS Capital, High Alpha, Benchmarkit 2025)", "GRR", latest & comparable & band & med),
    ("sc_band_q1", "NRR", "SaaS Capital 2025 lower quartile, ACV $25k to $50k", "NRR",
     (bench.source_id == "SC25") & band & (bench.statistic == "lower_quartile")),
    ("cm_logo", "Logo retention", "ChartMogul 2023 median, ARPA over $500 a month (low comparability)", "Logo retention",
     (bench.source_id == "CM23") & (bench.segment == "SMB and above: ARPA over $500 a month") & med),
    ("cm_logo_q3", "Logo retention", "ChartMogul 2023 upper quartile, ARPA over $500 a month (low comparability)", "Logo retention",
     (bench.source_id == "CM23") & (bench.segment == "SMB and above: ARPA over $500 a month") & (bench.statistic == "upper_quartile")),
]
rows = []
for key, measure, basis, metric, mask in specs:
    d = pick(mask & (bench.metric == metric))
    lo, hi, v = d.value.min(), d.value.max(), nb_vals[measure]
    rows.append({"key": key, "measure": measure, "benchmark_basis": basis, "benchmark_low": lo, "benchmark_high": hi,
                 "benchmark_rows": len(d), "sources": ", ".join(sorted(d.source_id.unique())), "northbridge_2025": v,
                 "position": "below" if v < lo else ("above" if v > hi else "within")})
comp = save_table(pd.DataFrame(rows), "4d_benchmark_comparison")
comp[["key", "measure", "benchmark_low", "benchmark_high", "sources", "northbridge_2025", "position"]].round(1)


,key,measure,benchmark_low,benchmark_high,sources,northbridge_2025,position
0,smb,NRR,94.0,107.0,"BM25, HA25, SC25",95.4,within
1,smb,NRR before price increase,94.0,107.0,"BM25, HA25, SC25",89.2,below
2,smb,GRR,83.0,92.0,"BM25, HA25, SC25",88.8,within
3,sc_smb,NRR,98.0,103.0,SC25,95.4,below
4,sc_smb,NRR before price increase,98.0,103.0,SC25,89.2,below
5,sc_smb,GRR,90.0,91.0,SC25,88.8,below
6,band,NRR,102.0,105.0,"BM25, HA25, SC25",95.4,below
7,band,GRR,88.0,92.0,"BM25, HA25, SC25",88.8,within
8,sc_band_q1,NRR,97.0,97.0,SC25,95.4,below
9,cm_logo,Logo retention,77.7,77.7,CM23,87.3,above


## 3. Charts

In [4]:
rep = nrr[nrr.basis == "reported"].set_index("year")
exp = nrr[nrr.basis == "excluding price increase"].set_index("year")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
for ax, col, title in zip(axes, ["nrr_pct", "grr_pct"], ["NRR", "GRR"]):
    x = np.arange(len(years)); w = 0.36
    b1 = ax.bar(x - w / 2, rep[col], w, color=ch.BLUE, label="Reported", edgecolor=ch.SURFACE, linewidth=1.5)
    b2 = ax.bar(x + w / 2, exp[col], w, color=ch.ORANGE, label="Excluding price increase", edgecolor=ch.SURFACE, linewidth=1.5)
    for bars in (b1, b2):
        for b in bars:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.6, f"{b.get_height():.1f}%", ha="center", fontsize=8, color=ch.TEXT)
    ax.axhline(100, color=ch.TEXT_2, lw=1, ls=(0, (3, 3)))
    ax.set_xticks(x); ax.set_xticklabels([str(y) for y in years])
    ax.set_ylim(70, 110); ax.yaxis.set_major_formatter(ch.pct_fmt())
    ax.set_title(f"{title} by year (2023 to 2025 only)")
    ax.set_xlabel("Year (December to December)")
axes[0].set_ylabel("% of opening-December MRR")
axes[0].legend(loc="lower left")
print(ch.save(fig, "4d_nrr_grr"))

outputs/charts/4d_nrr_grr.png


In [5]:
# Northbridge against the SMB bands of the three 2025 surveys: median dot, interquartile range where published
order = [s for s in bench[latest & comparable & smb].segment.drop_duplicates()]
def band_key(seg):
    b = seg.split(": ", 1)[1]
    nums = [float(x.split("k")[0].split(" ")[0]) for x in b.split("$")[1:]]
    return (b.startswith("ARPA"), 0.0 if "under" in b else nums[0], nums[-1])
order = sorted(order, key=band_key)
srcs = [("SC25", "SaaS Capital 2025"), ("HA25", "High Alpha 2025"), ("BM25", "Benchmarkit 2025")]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
for ax, metric, lines in ((axes[0], "NRR", [("NRR", "Northbridge 2025"), ("NRR before price increase", "before price increase")]),
                          (axes[1], "GRR", [("GRR", "Northbridge 2025")])):
    x = np.arange(len(order))
    for j, (sid, label) in enumerate(srcs):
        d = bench[(bench.source_id == sid) & (bench.metric == metric)].pivot(index="segment", columns="statistic", values="value")
        xs = [i + (j - 1) * 0.22 for i, s in enumerate(order) if s in d.index]
        ss = [s for s in order if s in d.index]
        if "lower_quartile" in d.columns:
            ax.vlines(xs, d.loc[ss, "lower_quartile"], d.loc[ss, "upper_quartile"], color=ch.SERIES[j], lw=2, alpha=0.45)
        ax.scatter(xs, d.loc[ss, "median"], s=36, color=ch.SERIES[j], edgecolor=ch.SURFACE, linewidth=1.5, zorder=3, label=label)
    for (measure, lab), ls in zip(lines, ["-", (0, (3, 3))]):
        v = nb_vals[measure]
        ax.axhline(v, color=ch.TEXT, lw=1.2, ls=ls)
        ax.text(len(order) - 0.45, v + 0.5, f"{lab} {v:.1f}%", ha="right", va="bottom", fontsize=8, color=ch.TEXT)
    ax.set_xticks(x); ax.set_xticklabels([s.replace("SMB: ACV ", "").replace(" to ", "-").replace("$", r"\$") for s in order], rotation=30, ha="right")
    ax.set_ylim(70, 120); ax.yaxis.set_major_formatter(ch.pct_fmt())
    ax.set_title(f"{metric}: SMB benchmarks by ACV band (US\\$)")
axes[0].set_ylabel("Median (dot) and interquartile range (line)")
axes[0].legend(loc="upper left")
print(ch.save(fig, "4d_benchmarks", source="Sources: SaaS Capital, 2025 B2B SaaS Retention Benchmarks; High Alpha, 2025 SaaS Benchmarks Report; "
              "Benchmarkit, 2025 B2B SaaS Performance Metrics Benchmarks (see docs/benchmarks.md). Northbridge: invoice ledger (synthetic data)."))


outputs/charts/4d_benchmarks.png


## 4. docs/benchmarks.md

Written from `benchmarks.csv`, `4d_benchmark_comparison.csv` and `4d_nrr_grr.csv`, so its figures cannot drift from the tables.

In [6]:
def fp(v):
    return f"{v:g}%"

def cell(d, seg, metric):
    s = d[(d.segment == seg) & (d.metric == metric)].set_index("statistic").value
    if "median" not in s:
        return "n/a"
    iqr = f" ({fp(s['lower_quartile'])} to {fp(s['upper_quartile'])})" if "lower_quartile" in s else ""
    return fp(s["median"]) + iqr

src_ids = list(bench.source_id.drop_duplicates())
meta = bench.drop_duplicates("source_id").set_index("source_id")
C = comp.set_index(["key", "measure"])
def c(key, measure, col="benchmark_low"):
    return C.loc[(key, measure), col]
def rng(key, measure):
    lo, hi = c(key, measure), c(key, measure, "benchmark_high")
    return fp(lo) if lo == hi else f"{fp(lo)} to {fp(hi)}"
NB = {k: v for k, v in nb_vals.items()}
WORD = {"below": "below", "within": "inside", "above": "above"}
def pos(key, measure):
    return WORD[c(key, measure, "position")]
uplift = f"{100 * (float(k[2025]) - 1):.0f}%"
def span(sid, metric):
    v = bench[(bench.source_id == sid) & (bench.metric == metric) & smb & med].value
    return f"{fp(v.min())} to {fp(v.max())}"
A = acv_tbl.iloc[0]
assert c("smb", "NRR", "position") == "within" and c("smb", "NRR before price increase", "position") == "below"
assert c("sc_smb", "NRR", "position") == "below" and c("sc_smb", "GRR", "position") == "below"

L = ["# SaaS retention benchmarks", "",
     "Generated by `notebooks/04d_nrr_grr.ipynb` from `data/reference/benchmarks_source.csv` (copied to `outputs/tables/benchmarks.csv`) "
     "and the 4d output tables. Do not edit by hand. Decisions D12 and D35 to D39 in `docs/decisions_log.md` record how the sources were chosen.", "",
     "Every benchmark figure below was read from the publisher's own report, opened on the date shown. Charts without printed values were not used, "
     "and no figure was taken from a secondary summary.", "",
     "## Sources", "",
     "| ID | Publisher and report | Edition | Sample and definition (the source's own) | Comparability | Accessed |",
     "| --- | --- | --- | --- | --- | --- |"]
for sid in src_ids:
    r = meta.loc[sid]
    L.append(f"| {sid} | {r.publisher}, [{r.title}]({r.url}) | {r.year} | {r.definition_notes} | {r.comparability} | {r.accessed_date} |")
L += ["", "Reports tried but not reachable from this session: KeyBanc Capital Markets and Sapphire Ventures' private SaaS survey (keybanc.com failed "
      "certificate verification at the proxy; sapphireventures.com and recurly.com were refused by the network policy). Bessemer's benchmark pages "
      "on bvp.com opened but the page tried returned 404, so Bessemer is not used.", "",
      "## SMB figures: median, with the interquartile range where the source publishes it", ""]
for metric in ("NRR", "GRR"):
    d = bench[bench.metric == metric]
    segs = sorted([s for s in d.segment.drop_duplicates() if s.startswith("SMB")], key=band_key)
    ids = [s for s in src_ids if (d.source_id == s).any()]
    L += [f"**{metric}**", "", "| Segment | " + " | ".join(ids) + " |", "| --- |" + " --- |" * len(ids)]
    for s in segs:
        L.append(f"| {s.split(': ', 1)[1]} | " + " | ".join(cell(d[d.source_id == i], s, metric) for i in ids) + " |")
    L.append("")
d = bench[bench.metric == "Logo retention"]
L += ["**Logo retention** (ChartMogul only; none of the surveys reports it by segment)", "", "| Segment | CM23 |", "| --- | --- |"]
for s in d.segment.drop_duplicates():
    L.append(f"| {s.split(': ', 1)[1]} | {cell(d, s, 'Logo retention')} |")
L += ["", "## Mid-market, enterprise and all-company figures (context only)", "",
      "| Segment | Source | NRR | GRR |", "| --- | --- | --- | --- |"]
for sid in src_ids:
    d = bench[bench.source_id == sid]
    for s in d.segment.drop_duplicates():
        if s.startswith("SMB"):
            continue
        L.append(f"| {s} | {sid} | {cell(d, s, 'NRR')} | {cell(d, s, 'GRR')} |")
L += ["", "## The SMB range adopted", "",
      f"- **NRR: SMB medians of {rng('smb', 'NRR')}. GRR: SMB medians of {rng('smb', 'GRR')}.** This is the lowest to the highest median for ACV bands under $50k "
      "in the latest edition of each source rated high or medium (SaaS Capital, High Alpha and Benchmarkit, all 2025 editions).",
      f"- **The closest like-for-like figure is SaaS Capital's: NRR {rng('sc_smb', 'NRR')}, GRR {rng('sc_smb', 'GRR')}** across its three SMB bands. Its definition is ours "
      "(December-to-December MRR, customers active in the opening December, price increases in NRR, GRR capped at opening MRR), and its sample is the largest.",
      f"- **Northbridge's own band.** Opening-December MRR × 12 over the customers active in December 2024 gives an average ACV of £{A.average_acv_gbp:,.0f}. "
      f"That sits in the $25k to $50k band at any exchange rate between ${A.gbp_usd_rate_for_25k_usd:.2f} and ${A.gbp_usd_rate_for_50k_usd:.2f} to the pound. "
      f"The three 2025 medians for that band are NRR {rng('band', 'NRR')} and GRR {rng('band', 'GRR')}. The average is lifted by six large accounts, "
      "so most customers sit in lower bands, which is why the full SMB range is the main comparison.",
      "- The 2023 SaaS Capital edition and ChartMogul's 2023 report are kept as context. The first is superseded by the 2025 edition; the second measures "
      "retention in a different way (below) and is rated low.", "",
      "Why the sources differ. High Alpha's bands under $10k ACV have its lowest medians, and its sample includes companies under $1M ARR. "
      f"Benchmarkit's SMB GRR medians ({span('BM25', 'GRR')}) sit at or below SaaS Capital's ({span('SC25', 'GRR')}), and Benchmarkit reports GRR drifting down over three years. "
      "ChartMogul's medians are far lower (for example GRR of "
      f"{fp(bench[(bench.source_id == 'CM23') & (bench.segment == 'SMB and above: ARPA over $500 a month') & (bench.metric == 'GRR') & med].value.iloc[0])} "
      "for ARPA over $500 a month) because it measures every company on its platform from billing data, including businesses from $300k ARR and 2022 data, "
      "rather than asking established companies to report their own figures. Survey medians are probably flattered by self-selection and self-reporting; "
      "ChartMogul's are probably depressed by small, early-stage companies.", "",
      "## Definition differences and what they mean for the comparison", "",
      "Our definitions (`docs/metric_definitions.md`): NRR is December MRR from the customers active in the opening December over their opening-December MRR; "
      "GRR caps each customer at its opening MRR; price increases count in NRR, with a sensitivity that removes them.", "",
      "| Point | SaaS Capital | High Alpha | Benchmarkit | ChartMogul | Effect on the comparison |",
      "| --- | --- | --- | --- | --- | --- |",
      "| Revenue basis | December MRR | Not stated | ARR | MRR | Small: for a subscription book, ARR is MRR × 12 |",
      "| Window | December to December | Annual, cohort | Calendar 2024 or last twelve months | 12 months, calendar 2022 | Small for the surveys; ChartMogul is three years older |",
      f"| Price increases in NRR | Included | Not stated | Not stated | Not stated | Northbridge's {uplift} rise in 2025 lifts its NRR; the before-price figure is the cautious comparison |",
      "| GRR cap | Each customer capped at opening MRR | Excludes expansion | Excludes expansion, includes churn and down-sell | Excludes expansion | Same in substance |",
      "| Statistic | Median of companies | Median of companies | Median and quartiles | Median and quartiles | One company against a distribution: position, not a pass mark |",
      "| Segment | Company ACV band (US$) | Average contract value (US$) | ACV band (US$) | ARPA per month (US$) | Northbridge's ACV is in £ and an average across very different customers |",
      "| Sample | Private B2B SaaS, over $1M ARR | 69% US; includes under $1M ARR | 583 participants | All ChartMogul users (B2B and B2C) | Largely US samples where stated; UK SMB software may differ |",
      "", "Comparability ratings: **high** where the definition matches ours (SaaS Capital); **medium** where the definition is close but the price treatment "
      "or revenue basis is not stated (High Alpha, Benchmarkit); **low** where the method or population differs materially (ChartMogul).", "",
      "## Where Northbridge sits (2025)", "",
      f"- **NRR {NB['NRR']:.1f}%** is {pos('smb', 'NRR')} the SMB range of medians ({rng('smb', 'NRR')}), {pos('sc_smb', 'NRR')} SaaS Capital's SMB medians ({rng('sc_smb', 'NRR')}), "
      f"{pos('band', 'NRR')} the medians for its own ACV band ({rng('band', 'NRR')}) and {pos('sc_band_q1', 'NRR')} SaaS Capital's lower quartile for that band ({rng('sc_band_q1', 'NRR')}).",
      f"- **NRR before the price increase, {NB['NRR before price increase']:.1f}%**, is {pos('smb', 'NRR before price increase')} the SMB range of medians ({rng('smb', 'NRR')}).",
      f"- **GRR {NB['GRR']:.1f}%** is {pos('smb', 'GRR')} the SMB range of medians ({rng('smb', 'GRR')}), {pos('sc_smb', 'GRR')} SaaS Capital's SMB medians ({rng('sc_smb', 'GRR')}) "
      f"and {pos('band', 'GRR')} the range of medians for its own band ({rng('band', 'GRR')}).",
      f"- **Logo retention {NB['Logo retention']:.1f}%** is {pos('cm_logo_q3', 'Logo retention')} ChartMogul's upper quartile for ARPA over $500 a month ({rng('cm_logo_q3', 'Logo retention')}), "
      "but that source is rated low and no survey reports logo retention by segment, so this is not used in the findings.", "",
      "In short: on the definition that matches ours, Northbridge's revenue retention is below the typical SMB software company, and before price increases "
      "it is below the whole SMB range. The comparison supports the existing High rating of the retention finding (F03) rather than changing it. "
      "Benchmarks are medians of survey respondents, largely US-based where stated, so they place Northbridge in a distribution; they do not set a pass mark.", ""]
(DOCS / "benchmarks.md").write_text("\n".join(L), encoding="utf-8")
print(f"wrote docs/benchmarks.md ({len(L)} lines)")


wrote docs/benchmarks.md (116 lines)


In [7]:
nrr

,year,basis,uplift_factor,opening_mrr,closing_mrr_same_customers,closing_mrr_capped,nrr_pct,grr_pct,customers_opening,customers_retained
0,2023,reported,1.00,2417140.30,2.314809e+06,2.284726e+06,95.766411,94.521877,1010,920
1,2023,excluding top 5 customers,1.00,1711723.63,1.609392e+06,1.579310e+06,94.021711,92.264294,1005,915
2,2023,excluding price increase,1.00,2417140.30,2.314809e+06,2.284726e+06,95.766411,94.521877,1010,920
3,2023,excluding top 5 and price increase,1.00,1711723.63,1.609392e+06,1.579310e+06,94.021711,92.264294,1005,915
4,2024,reported,1.05,2731778.55,2.697551e+06,2.538154e+06,98.747049,92.912156,1205,1073
5,2024,excluding top 5 customers,1.05,2026361.88,1.975238e+06,1.845279e+06,97.477071,91.063663,1200,1068
6,2024,excluding price increase,1.05,2731778.55,2.569096e+06,2.531658e+06,94.044808,92.674359,1205,1073
7,2024,excluding top 5 and price increase,1.05,2026361.88,1.881179e+06,1.843742e+06,92.835306,90.987776,1200,1068
8,2025,reported,1.07,3124736.34,2.982357e+06,2.773799e+06,95.443477,88.769051,1343,1172
9,2025,excluding top 5 customers,1.07,2402423.85,2.337281e+06,2.153774e+06,97.288444,89.650034,1338,1167
